## 1. Konfiguracja stałych

### **LLM_MODELS** - modele LLM do wykorzystania
- **bielik** - model __Bielik 11B v3.0-Instruct__ - polish text
- **gemma4:small** - model __Gemma4 8B E4B - text, image, reasoning
- **gemma4:large** - model Gemma4 26B MoE - text, image, reasoning, Frontier like

### **EMB_MODELS** - model do embedding dostarczony w standardzie OpenAI Client
- **stella-embeddings** - model embeddings Stella 8B
- **polish-reranker** - model reranking for Polish Texts


In [ ]:
BASE_URL = "http://engine.kin.tu.kielce.pl:32597/v1"
API_KEY = '<TWOJ_API_KEY>'
LLM_MODELS=['bielik', 'gemma4:small', 'gemma4:large']

EMB_MODEL = "stella-embeddings"
RERANK_MODEL = "polish-reranker"


## 2. Konfiguracja klienta

#### Proste pobranie wiadomości tekstowej (zamiast odwolania do tablicy, podajemy nazwe modelu)

In [ ]:
from openai import OpenAI

client = OpenAI(base_url=BASE_URL, api_key=API_KEY)

In [ ]:
response = client.chat.completions.create(model=LLM_MODELS[0], messages=[{"role": "user", "content": "Jaka jest stolica Hawajów?"}])

print(response.choices[0].message.content)

## 3. Przykłady użycia udostepnionych modeli

### 3.1 Użycie modelu embeddingowego

In [ ]:
input = 'To jest tekst do zamiany na wektory'

response = client.embeddings.create(
            input=input,
            model=EMB_MODEL
        )
print(f'Rozmiar wektora: {len(response.data[0].embedding)}')
print(f'Liczba wektorów: {len(response.data)}')

### 3.2 Użycie modelu rerankingowego

In [ ]:
import requests

url = f"{client.base_url}rerank"

headers = {
       "Authorization": f"Bearer {client.api_key}",
        "Content-Type": "application/json"
   }

query = "Jakim kablem zasilany jest laptop?"
docs = [
    "Laptop wyposażono w złącze HDMI, aby przesłać obraz na zewnętrzny monitor.",
    "Stary czajnik elektryczny wymaga kabla zasilającego typu koniczynka.",
    "Laptop jest podłączony do sieci za pomocą oryginalnego zasilacza USB-C o mocy 96W.",
    "Wczorajsza burza spowodowała krótką przerwę w dostawie prądu na całym osiedlu."
]

payload = {
        "model": RERANK_MODEL,
        "query": query,
        "documents": docs
    }
response = requests.post(url, json=payload, headers=headers)
rerank_results = response.json()
for result in rerank_results['results']:
    doc_idx = result['index']
    score = result['relevance_score']
    print(f"Score: {score:.4f} | Treść: {docs[doc_idx]}")

### 3.3 Przykładowe użycie modelu embeddingowego z zapisem dokumentow do bazy ChromaDB

In [ ]:
import glob
from langchain_community.document_loaders import DirectoryLoader, TextLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
import os


folders = glob.glob("docs")

documents = []
for folder in folders:
    doc_type = os.path.basename(folder)
    loader = DirectoryLoader(folder, glob="**/*.md", loader_cls=TextLoader, loader_kwargs={'encoding': 'utf-8'})
    folder_docs = loader.load()
    for doc in folder_docs:
        doc.metadata["doc_type"] = doc_type
        documents.append(doc)


text_splitter = RecursiveCharacterTextSplitter(chunk_size=400, chunk_overlap=100)
chunks = text_splitter.split_documents(documents)

print(f"Loaded {len(documents)} documents")
print(f"Divided into {len(chunks)} chunks")
print(f"First chunk:\n\n{chunks[0]}")

In [ ]:
from chromadb import EmbeddingFunction, Documents, Embeddings, Settings
import chromadb

db_name = "vector_db"

class StellaEmbeddingFunction(EmbeddingFunction):

    def __init__(self):
        pass

    def __call__(self, input: Documents) -> Embeddings:
        response = client.embeddings.create(
            input=input,
            model=EMB_MODEL
        )
        return [data.embedding for data in response.data]

my_embedding_function = StellaEmbeddingFunction()

db_client = chromadb.PersistentClient(
    path=db_name,
    settings=Settings(allow_reset=True)
)

try:
    db_client.reset()
    print("Reset bazy danych.")
except ValueError:
    print("Resetowanie nie jest włączone w ustawieniach.")

collection = db_client.get_or_create_collection(
    name="my_collection",
    embedding_function=my_embedding_function
)

In [ ]:
documents = [c.page_content for c in chunks]
metadatas = [c.metadata for c in chunks]
ids = [f"id_{i}" for i in range(len(chunks))]

batch_size = 50

for i in range(0, len(documents), batch_size):
    collection.add(
        documents=documents[i : i + batch_size],
        metadatas=metadatas[i : i + batch_size],
        ids=ids[i : i + batch_size]
    )

print(f"Ingestion complete. Collection '{collection.name}' now contains {collection.count()} records.")

In [ ]:
count = collection.count()

sample_embedding = collection.get(limit=1, include=["embeddings"])["embeddings"][0]
dimensions = len(sample_embedding)
print(f"There are {count:,} vectors with {dimensions:,} dimensions in the vector store")

## 4. Przykład przesłania zdjęcia do modelu Vision

In [ ]:
import base64
from PIL import Image
import io

def process_image_for_api(image_path, max_size=(640, 480)):

    with Image.open(image_path) as img:
        img = img.convert("RGB")
        img.thumbnail(max_size)

        buffered = io.BytesIO()
        img.save(buffered, format="JPEG", quality=85)
        return base64.b64encode(buffered.getvalue()).decode('utf-8')

In [ ]:
image_path = "img/example.png"
base64_image = process_image_for_api(image_path)

response = client.chat.completions.create(
    model="gemma4:small",
    messages=[
        {
            "role": "user",
            "content": [
                {"type": "text",
                 "text": "Describe this image in MAX 3 SENTENCES."
                 },
                {
                    "type": "image_url",
                    "image_url": {
                        "url": f"data:image/jpeg;base64,{base64_image}"
                    },
                },
            ],
        }
    ]
)

print(f"Model response:\n{response.choices[0].message.content}")